# Food festival

In [ ]:
import Pkg
Pkg.add("HiGHS")
Pkg.add("JuMP")


In [ ]:
using JuMP, HiGHS

include("Data/FoodFestival_data.jl")
println(Conflict) # Binary variable
println(Shifts)
println(S)
println(ConflictingShifts)


########## ---------- Models ---------- ##########
model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)

########## ---------- Variables ---------- ##########
@variable(model, x[1:S, 1:S], Bin) # Schedule
@variable(model, y[1:S], Bin) # Workers

########## ---------- Objective ---------- ##########
@objective(model, Min, sum(y[i] for i in 1:S))

########## ---------- Constraint ---------- ##########
# Define y_i as a person working a shift
@constraint(model, [i in 1:S, j in 1:S], x[i,j] <= y[i])

# All rows in x must be 1
@constraint(model, [j in 1:S], sum(x[i,j] for i in 1:S) == 1 )

# If a worker works a shift with conflict, the sum of the shift + it's conflictis should be 1
@constraint(model, [i in 1:S, s in 1:S, t in 1:S; Conflict[s,t] == 1],
    x[i,s] + x[i,t] <= 1
)

########## ---------- Result ---------- ##########
optimize!(model)
println("Optimal solution:")
println(objective_value(model))
println(value.(y))

for i in 1:S
    println(value.(x[i, :]))
end

Int8[0 1 0 1 0 1 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 1 0 0 1; 1 0 1 0 0 0 1 0 1 0 0 0 0 1 0 1 1 0 0 0 0 0 1 0 0; 0 1 0 1 0 0 0 0 1 1 1 0 0 0 0 0 0 1 1 0 0 1 1 0 0; 1 0 1 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 1 1 1 0 0 1 1; 0 0 0 0 0 1 0 0 0 1 1 0 1 0 1 1 0 0 0 0 1 1 0 0 0; 1 0 0 0 1 0 0 1 0 0 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0; 0 1 0 0 0 0 0 0 1 1 0 0 0 1 1 0 0 0 0 0 1 1 0 0 1; 0 0 0 1 0 1 0 0 1 1 1 0 0 0 1 1 0 0 0 0 0 0 0 1 0; 0 1 1 1 0 0 1 1 0 0 0 1 1 0 1 0 0 0 1 0 1 0 0 1 0; 0 0 1 1 1 0 1 1 0 0 1 0 0 1 1 1 0 0 1 0 0 1 0 0 0; 1 0 1 0 1 1 0 1 0 1 0 1 1 0 1 0 0 0 1 1 1 0 0 0 1; 0 0 0 0 0 1 0 0 1 0 1 0 0 1 1 1 1 1 0 0 0 0 1 0 0; 0 0 0 0 1 1 0 0 1 0 1 0 0 1 0 1 0 0 1 0 0 0 1 0 1; 0 1 0 0 0 1 1 0 0 1 0 1 1 0 1 0 1 1 0 0 0 1 0 0 0; 0 0 0 0 1 1 1 1 1 1 1 1 0 1 0 1 1 0 0 0 0 0 1 1 1; 0 1 0 0 1 1 0 1 0 1 0 1 1 0 1 0 0 1 1 1 1 0 0 1 0; 1 1 0 0 0 1 0 0 0 0 0 1 0 1 1 0 0 0 1 0 0 1 0 1 0; 0 0 1 0 0 1 0 0 0 0 0 1 0 1 0 1 0 0 0 1 1 0 0 0 0; 0 0 1 1 0 1 0 0 1 1 1 0 1 0 0 1 1 0 0 0 0 0 1 1 1; 0 0 0 1 0 0 0 0 0 0 1 0 0 

# Wedding planner

• Each guest should sit at a table <br>
• There is a limit of 9 people at each table <br>
• Couples that attend the wedding should sit at the same table

In [29]:
using JuMP, HiGHS
include("Data/WeddingData20.jl")

model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)


@variable(model, x[1:G, 1:T], Bin) # Tables

@objective(model, Max, 1)

########## ---------- Constraint ---------- ##########
# All guest needs to be assigned a seat
@constraint(model, [g in 1:G], sum(x[g,t] for t in 1:T) == 1)

# A table consist of maximum 9 people
@constraint(model, [t in 1:T],
    sum(x[g,t] for g in 1:G) <= 9
)

# Couples should sit together
@constraint(model, [t in 1:T, g1 in 1:G, g2 in 1:G; Couple[g1, g2] == 1],
    x[g1, t] == x[g2, t]
)


########## ---------- Result ---------- ##########
optimize!(model)
println("Optimal solution:")
println(objective_value(model))

for i in 1:T
    println(value.(x[:, i]))
    println(sum(value.(x[:, i])))
end

Optimal solution:
1.0
[0.0, -0.0, -0.0, 0.0, -0.0, 0.0, -0.0, 0.0, -0.0, -0.0, 0.0, -0.0, 0.0, -0.0, 0.0, -0.0, 1.0, 1.0, 0.0, -0.0]
2.0
[0.0, -0.0, 1.0, 1.0, 1.0, 0.0, -0.0, 1.0, 1.0, -0.0, 1.0, 1.0, 0.0, -0.0, 0.0, -0.0, -0.0, -0.0, 1.0, 1.0]
9.0
[1.0, 1.0, -0.0, 0.0, -0.0, 1.0, 1.0, 0.0, -0.0, 1.0, 0.0, -0.0, 1.0, 1.0, 1.0, 1.0, -0.0, -0.0, 0.0, -0.0]
9.0


Add the following objective function to your model:
Maximize the number of shared interests for persons sitting at the same table

In [49]:
using JuMP, HiGHS
include("Data/WeddingData20.jl")

model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)


@variable(model, x[1:G, 1:T], Bin) # Tables
@variable(model, y[1:G, 1:G, 1:T], Bin) # Check if they sit at the same table

@objective(model, Max, 
    sum(y[g1, g2, t] * SharedInterests[g1,g2] for g1 in 1:G, g2 in 1:G, t in 1:T) / 2
)



########## ---------- Constraint ---------- ##########
# All guest needs to be assigned a seat
@constraint(model, [g in 1:G], sum(x[g,t] for t in 1:T) == 1)

# A table consist of maximum 9 people
@constraint(model, [t in 1:T],
    sum(x[g,t] for g in 1:G) <= 9
)

# Couples should sit together
@constraint(model, [t in 1:T, g1 in 1:G, g2 in 1:G; Couple[g1, g2] == 1],
    x[g1, t] == x[g2, t]
)

########## ---------- Constraint : Sit togheter ---------- ##########
# let y[g1, g2, T] == 1 if guest sit at the same table
#
# Logical AND:
# v <= x
# v <= y
# v >= x + y - 1
@constraint(model, [t in 1:T, g1 in 1:G, g2 in 1:G],
    y[g1, g2, t] <= x[g1,t] 
)

@constraint(model, [t in 1:T, g1 in 1:G, g2 in 1:G],
    y[g1, g2, t] <= x[g2,t] 
)

@constraint(model, [t in 1:T, g1 in 1:G, g2 in 1:G],
    y[g1, g2, t] >= x[g1,t] + x[g2,t] - 1
)


########## ---------- Result ---------- ##########
optimize!(model)
println("Optimal solution:")
println(objective_value(model))

for i in 1:T
    println(value.(x[:, i]))
    println(sum(value.(x[:, i])))
end

Optimal solution:
67.0
[1.0, 1.0, -0.0, 0.0, -0.0, 0.0, -0.0, 0.0, -0.0, -0.0, 0.0, -0.0, 0.0, -0.0, 0.0, -0.0, -0.0, 1.0, 0.0, -0.0]
3.0
[0.0, -0.0, 1.0, 0.0, 0.0, 0.0, -0.0, 0.0, -0.0, 1.0, 1.0, 1.0, 0.0, -0.0, 1.0, 1.0, -0.0, -0.0, 1.0, 1.0]
8.0
[0.0, -0.0, -0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, -0.0, 1.0, 1.0, 0.0, 0.0, 1.0, -0.0, 0.0, 0.0]
9.0


Now consider a different objective function: For each table, penalize the number of
males and the number of females that are in excess of two. As an example, a table
with seven males and two females has an excess of five males and incurs a penalty
of three. Minimize the sum of penalties for all tables.

In [ ]:
using JuMP, HiGHS
include("Data/WeddingData20.jl")

model = Model(HiGHS.Optimizer)
set_optimizer_attribute(model, "log_to_console", false)


@variable(model, x[1:G, 1:T], Bin) # Tables
@variable(model, y[1:T] >= 0 ) # Slack to enforce balance constratint

@objective(model, Min, 
    sum(y[t] for t in 1:T)
)



########## ---------- Constraint ---------- ##########
# All guest needs to be assigned a seat
@constraint(model, [g in 1:G], sum(x[g,t] for t in 1:T) == 1)

# A table consist of maximum 9 people
@constraint(model, [t in 1:T],
    sum(x[g,t] for g in 1:G) <= 9
)

# Couples should sit together
@constraint(model, [t in 1:T, g1 in 1:G, g2 in 1:G; Couple[g1, g2] == 1],
    x[g1, t] == x[g2, t]
)

########## ---------- Constraint : Male/Female balance ---------- ##########

@constraint(model, [t in 1:T],
    sum(x[g, t] * Male[g] - x[g, t] * Female[g] for g in 1:G) - y[t] == 0
)
@constraint(model, [t in 1:T],
    sum(x[g, t] * Female[g] - x[g, t] * Male[g] for g in 1:G) - y[t] == 0
)



########## ---------- Result ---------- ##########
optimize!(model)
println("Optimal solution:")
println(objective_value(model))

for i in 1:T
    println(value.(x[:, i]))
    println(sum(value.(x[:, i])))
    println(value.(y[i]))
end

Optimal solution:
0.0
[0.0, -0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, -0.0, 1.0, 0.0, -0.0, 1.0, 1.0, 0.0, -0.0, -0.0, -0.0, 0.0, -0.0]
8.0
0.0
[0.0, -0.0, -0.0, 0.0, -0.0, 0.0, -0.0, 1.0, 1.0, -0.0, 0.0, -0.0, 0.0, -0.0, 0.0, -0.0, 1.0, 1.0, 1.0, 1.0]
6.0
0.0
[1.0, 1.0, -0.0, 0.0, -0.0, 0.0, -0.0, 0.0, -0.0, -0.0, 1.0, 1.0, 0.0, -0.0, 1.0, 1.0, -0.0, -0.0, 0.0, -0.0]
6.0
0.0


: 